# 🧠 Brain Tumor MRI Classification — Exploratory Data Analysis

**Dataset:** [Brain Tumor MRI Dataset on Kaggle](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)  
**Classes:** Glioma | Meningioma | Pituitary | No Tumor  
**Goal:** Multi-class CNN classifier to detect tumor type from MRI scans


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
import cv2
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor'] = '#161b22'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

print('✅ Libraries loaded successfully')

In [ ]:
# ─── Dataset Paths ───────────────────────────────────────────────
# After downloading from Kaggle, set this path:
DATA_DIR = Path('../data')   # update if needed
TRAIN_DIR = DATA_DIR / 'Training'
TEST_DIR  = DATA_DIR / 'Testing'

CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
CLASS_LABELS = {c: i for i, c in enumerate(CLASSES)}
print('Classes:', CLASS_LABELS)

In [ ]:
# ─── Count images per class ───────────────────────────────────────
def count_images(base_dir, classes):
    counts = {}
    for cls in classes:
        path = base_dir / cls
        if path.exists():
            counts[cls] = len(list(path.glob('*.jpg')) + list(path.glob('*.png')))
        else:
            counts[cls] = 0
    return counts

train_counts = count_images(TRAIN_DIR, CLASSES)
test_counts  = count_images(TEST_DIR, CLASSES)

df_dist = pd.DataFrame({'Train': train_counts, 'Test': test_counts})
print(df_dist)
print(f"\nTotal Training Images : {sum(train_counts.values())}")
print(f"Total Testing Images  : {sum(test_counts.values())}")

In [ ]:
# ─── Class Distribution Plot ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff']

for ax, (split, counts) in zip(axes, [('Train', train_counts), ('Test', test_counts)]):
    bars = ax.bar(counts.keys(), counts.values(), color=colors, edgecolor='none', width=0.6)
    ax.set_title(f'{split} Set Distribution', fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Tumor Class')
    ax.set_ylabel('Image Count')
    for bar, val in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(val), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved: results/class_distribution.png')

In [ ]:
# ─── Sample Images per Class ──────────────────────────────────────
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('Sample MRI Images per Class', fontsize=16, fontweight='bold', y=1.01)

for row, cls in enumerate(CLASSES):
    cls_path = TRAIN_DIR / cls
    images = list(cls_path.glob('*.jpg'))[:4] + list(cls_path.glob('*.png'))[:4]
    images = images[:4]
    for col, img_path in enumerate(images):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        axes[row][col].imshow(img, cmap='inferno')
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_title(cls.upper(), fontsize=10, fontweight='bold',
                                     loc='left', color='#58a6ff')

plt.tight_layout()
plt.savefig('../results/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('🖼️  Saved: results/sample_images.png')

In [ ]:
# ─── Image Size Analysis ──────────────────────────────────────────
widths, heights = [], []
for cls in CLASSES:
    for img_path in list((TRAIN_DIR / cls).glob('*.jpg'))[:50]:
        img = Image.open(img_path)
        w, h = img.size
        widths.append(w)
        heights.append(h)

print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')
print(f'\n→ We will resize all images to 224×224 for our CNN')

In [ ]:
# ─── Pixel Intensity Distribution ────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
fig.suptitle('Pixel Intensity Distribution per Class', fontsize=13, fontweight='bold')

for ax, (cls, color) in zip(axes, zip(CLASSES, colors)):
    all_pixels = []
    for img_path in list((TRAIN_DIR / cls).glob('*.jpg'))[:30]:
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (224, 224))
        all_pixels.extend(img.flatten().tolist())
    ax.hist(all_pixels, bins=50, color=color, alpha=0.85, edgecolor='none')
    ax.set_title(cls, fontweight='bold')
    ax.set_xlabel('Pixel Value')

axes[0].set_ylabel('Frequency')
plt.tight_layout()
plt.savefig('../results/pixel_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved: results/pixel_distribution.png')

## ✅ EDA Summary

| Finding | Detail |
|---|---|
| Total Classes | 4 (Glioma, Meningioma, No Tumor, Pituitary) |
| Class Imbalance | Slight imbalance — handled via class weights |
| Image Sizes | Varying — will normalize to 224×224 |
| Preprocessing Needed | Resize, Normalize (0–1), Augmentation |

➡️ Proceed to **Notebook 02** for model training.